Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* Actual ResNet Included
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 80 blocks (MaxPool2D in each second block)

In [ ]:
out_channels = 8
size = 64

model_blocks = [
    nn.Conv2d(3, out_channels, 3, 1, 1),
    nn.BatchNorm2d(out_channels),
    nn.ReLU()
]

for stage in range(5):

    for i in range(20):

        conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        bn = nn.BatchNorm2d(out_channels)

        model_blocks.append(ResidualBlock([conv, bn, nn.ReLU()]))

    if stage < 3:
        model_blocks.append(nn.MaxPool2d(2, 2))
        size //= 2
    if stage<4:
        model_blocks.extend([
            nn.Conv2d(out_channels, out_channels*2, 3, 1, 1),
            nn.BatchNorm2d(out_channels*2),
            nn.ReLU()
    ])

        out_channels *= 2

print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment7/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment7/tables/training_metrics.csv", index=False)

In [ ]:
filtered_df = df[
    (df["val_accuracy"] >= 0.8) &
    (df["val_f1"] >= 0.8)
].to_dict()

high_accuracy_df = plot_training_metrics(filtered_df)

### Training/Validation Trend (100 epochs)
* Training loss consistently decreased from 1.0852 (epoch 1) to 0.1196 (epoch 100), showing continuous optimization and learning.
Training accuracy, precision, recall, and F1-score improved steadily, reaching above 97% after epoch 90, indicating strong fitting on the training data.
* Validation performance improved rapidly during early epochs, increasing from around 40% accuracy in epoch 1 to above 80% accuracy after epoch 70.
* Validation metrics showed fluctuations throughout training, with several drops after strong epochs, indicating some instability and possible overfitting tendency.
* The best validation improvement occurred around epochs 85–97, where validation accuracy, precision, recall, and F1 remained consistently high.
* After epoch 97, validation performance slightly decreased while training metrics continued increasing, suggesting the beginning of overfitting.
* No training metric reached 1.00, so no epoch needed to be rejected due to perfect training scores.
* The gap between training and validation performance increased in later epochs, showing that the model continued memorizing training patterns more than improving generalization.

The model learned effectively throughout training, with strong improvement in both training and validation metrics. The first 50 epochs showed consistent learning, while later epochs showed higher training accuracy but more unstable validation behaviour. Validation performance peaked near the end of training, but after epoch 97 the model started showing signs of overfitting because training metrics continued increasing while validation metrics declined. Epoch 97 provides the best balance between high validation performance and avoiding unnecessary additional fitting.

<b>Best Epoch 97</b>

<b>Loss</b>
* Train Loss = 0.1383646849
* Valid Loss = 0.5174597586

<b>Training Metrics</b>
* Train Accuracy = 0.9763705134
* Train Precison = 0.9769217968
* Train Recall = 0.9766728282
* Train F1 = 0.9767453074

<b>Validation Accuracy</b>
* Validation Accuracy = 0.8340708017
* Validation Precision = 0.8370292187
* Validation Recall = 0.8348357081
* Validation F1 = 0.8356957436

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment7"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment7/tables/test_metrics.csv", index=False)

### Test Performance Trend (100 epochs)
* Test loss generally decreases as training progresses, showing improved model confidence.
* Test accuracy increases from around 35% in early epochs to above 70% in later epochs.
* Precision, recall, and F1-score improve together, indicating better classification performance across classes.
* Early test epochs show unstable predictions and weak class separation.
* Around epoch 100–150, the model starts achieving consistent improvement.
* Around epoch 200–260, the model reaches its strongest test performance.
* The highest improvement occurs near the final part of testing, with accuracy stabilizing around 70–74%.
* Test metrics fluctuate between epochs, indicating that some checkpoints generalize better than others.
* The model does not show a major decline after reaching high performance.
* The best checkpoint is not the final epoch, suggesting that saving the best-performing epoch is preferable.

The test performance shows that the CNN learned useful features progressively, with major improvements occurring during the first 60 epochs. After this point, the model entered a stable convergence phase where test metrics fluctuated around high performance values rather than continuously increasing. The highest generalization performance was achieved near the end of training, with epoch 97 providing the best combination of accuracy, precision, recall, and F1-score. Unlike the training metrics, the test metrics did not show signs of overfitting because performance remained strong and stable.

<b>Best Epoch 97</b>

* Loss = 0.4317497499
* Accuracy = 0.8596491218
* Precision = 0.8603258133
* Recall = 0.8618202209
* F1-Score = 0.8597231507